# PrintScale cloud GPU server (AuraSR v2)

Turns this notebook into a private upscaling server for the **PrintScale Android app**. The app sends its prepared artwork here, AuraSR v2 upscales it 4× on this GPU, and the app downloads the result. Print sizing, DPI, colour profile, history and export all happen in the app.

**How to connect:** run every cell. The last cell shows a link and a QR code. Scan the QR code with your phone's camera and tap **Open in the PrintScale app**, or copy the link into the app (Home > Cloud GPU > paste link).

* The link is new each time the notebook starts and contains a private key; anyone with the full link can use your GPU, so do not share it.
* Keep this tab open while you work. The server stops when the notebook stops or the session ends.
* Uploads are kept only while a job runs and are deleted after download (or after one hour).
* **Free-tier rules:** Colab's free tier may disconnect notebooks that serve outside traffic; Kaggle sessions stop after 12 hours. If the app says it cannot reach the server, restart this notebook and connect with the new link.
* AuraSR v2 weights are CC BY-SA 4.0 (commercial use allowed with credit: "Upscaled with AuraSR v2 by fal.ai").

**Before you start (right panel):** Settings > Accelerator > **GPU T4 x2** and Settings > **Internet on**. Then Run All.

In [ ]:
# 1. Check the GPU
import os, subprocess, torch
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}, {props.total_memory / 2**30:.0f} GB. AuraSR will run on it.")
elif os.environ.get("PRINTSCALE_ALLOW_CPU") == "1":
    print("No GPU: running on CPU (test mode, very slow).")
else:
    raise RuntimeError("No GPU. In Kaggle: Settings (right panel) > Accelerator > GPU T4 x2. Also turn Internet on.")

In [ ]:
# 2. Install AuraSR and the QR code helper (about 1 minute)
%pip install -q aura-sr==0.0.4 einops safetensors qrcode

**3. Server code.** No changes needed: run the next cells.

In [ ]:
%%writefile printscale_aurasr.py
# PrintScale AuraSR: batch upscaling of artwork for print-on-demand.
# Model: fal/AuraSR-v2 (GigaGAN-based 4x upscaler for AI-generated images), licence CC BY-SA 4.0.
# This file is embedded verbatim in the Colab and Kaggle notebooks.

import io
import json
import math
import os
import time
import zipfile
from dataclasses import dataclass, field

import numpy as np
from PIL import Image, ImageDraw, ImageOps

Image.MAX_IMAGE_PIXELS = None  # print files are large on purpose

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}

# Short side x long side in inches.
PRESETS = {
    "5x7": (5, 7), "8x10": (8, 10), "8x12": (8, 12), "10x12": (10, 12),
    "11x14": (11, 14), "12x16": (12, 16), "12x18": (12, 18), "16x20": (16, 20),
    "18x24": (18, 24), "20x24": (20, 24), "20x30": (20, 30), "24x36": (24, 36),
    "30x40": (30, 40), "36x48": (36, 48),
}


# ----------------------------------------------------------------------------- model

def load_aurasr(model_id="fal/AuraSR-v2"):
    """Loads AuraSR on the GPU if there is one, otherwise on the CPU (very slow)."""
    import torch
    from aura_sr import AuraSR
    from huggingface_hub import snapshot_download
    from safetensors.torch import load_file

    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda":
        torch.backends.cudnn.benchmark = True  # tiles are all 64 px, so cuDNN can pick the fastest kernels once
    path = snapshot_download(model_id, allow_patterns=["config.json", "model.safetensors"])
    with open(os.path.join(path, "config.json")) as f:
        config = json.load(f)
    model = AuraSR(config, device=device)  # the package's from_pretrained always assumes CUDA
    model.upsampler.load_state_dict(load_file(os.path.join(path, "model.safetensors")), strict=True)
    model.upsampler.eval()
    return model, device


def _ai_block(model, rgb: Image.Image, seed: int, batch: int = 8) -> np.ndarray:
    """4x one block. Pads to whole 64 px tiles with edge pixels so small or odd sizes work."""
    import torch
    tile = model.input_image_size
    w, h = rgb.size
    pw = math.ceil(w / tile) * tile
    ph = math.ceil(h / tile) * tile
    arr = np.asarray(rgb)
    if (pw, ph) != (w, h):
        arr = np.pad(arr, ((0, ph - h), (0, pw - w), (0, 0)), mode="edge")
    torch.manual_seed(seed)  # the GAN takes a noise vector; fixed seed = repeatable output
    out = model.upscale_4x_overlapped(Image.fromarray(arr), max_batch_size=batch)
    return np.asarray(out, dtype=np.float32)[: h * 4, : w * 4]


def default_batch():
    """Tiles per GPU call. AuraSR's own default of 8 leaves a T4 mostly idle."""
    import torch
    if not torch.cuda.is_available():
        return 8
    gb = torch.cuda.get_device_properties(0).total_memory / 2**30
    return 64 if gb >= 20 else 32 if gb >= 12 else 16


def _blocks(w, h, block, overlap):
    if w <= block and h <= block:
        return [(0, 0, w, h)]
    step = block - overlap
    xs = list(range(0, max(w - overlap, 1), step))
    ys = list(range(0, max(h - overlap, 1), step))
    return [(x0, y0, min(x0 + block, w), min(y0 + block, h)) for y0 in ys for x0 in xs]


def _batches_for(bw, bh, tile, batch):
    """GPU calls AuraSR makes for one block: a normal pass plus a half-tile-offset pass."""
    pw, ph = math.ceil(bw / tile) * tile, math.ceil(bh / tile) * tile
    t1 = (pw // tile) * (ph // tile)
    t2 = ((pw + tile) // tile) * ((ph + tile) // tile)
    return math.ceil(t1 / batch) + math.ceil(t2 / batch)


def ai_upscale_4x(model, rgb: Image.Image, block=1024, overlap=64, seed=0, progress=None, batch=8,
                  cancelled=None) -> Image.Image:
    """
    4x a whole image in overlapping blocks and feather them together, so memory stays bounded
    for any input size (AuraSR keeps its whole output as float tensors). progress(done, total)
    is called after every GPU batch; cancelled() is checked at the same points.
    """
    w, h = rgb.size
    tile = model.input_image_size
    # Keep every block on AuraSR's own 64 px tile grid so blocks match whole-image inference.
    overlap = max(tile, round(overlap / tile) * tile)
    block = max(overlap + tile, round(block / tile) * tile)
    blocks = _blocks(w, h, block, overlap)
    total = sum(_batches_for(x1 - x0, y1 - y0, tile, batch) for x0, y0, x1, y1 in blocks)
    count = [0]

    def on_batch(*_):
        count[0] += 1
        if cancelled is not None and cancelled():
            raise InterruptedError("cancelled")
        if progress:
            progress(min(count[0], total), total)

    hook = model.upsampler.register_forward_hook(on_batch)
    try:
        if len(blocks) == 1:
            out = _ai_block(model, rgb, seed, batch)
            return Image.fromarray(np.clip(out + 0.5, 0, 255).astype(np.uint8))

        acc = np.zeros((h * 4, w * 4, 3), np.float32)
        wsum = np.zeros((h * 4, w * 4, 1), np.float32)
        ramp = overlap * 4
        for x0, y0, x1, y1 in blocks:
            part = _ai_block(model, rgb.crop((x0, y0, x1, y1)), seed, batch)
            bh, bw = part.shape[:2]

            def axis(n, at_start, at_end):
                # linear cross-fade over the overlap; edges of the image keep full weight
                v = np.ones(n, np.float32)
                r = min(ramp, n // 2)
                if r > 0:
                    up = (np.arange(r, dtype=np.float32) + 0.5) / r
                    if not at_start:
                        v[:r] = up
                    if not at_end:
                        v[n - r:] = np.minimum(v[n - r:], up[::-1])
                return v

            wt = (axis(bh, y0 == 0, y1 == h)[:, None] * axis(bw, x0 == 0, x1 == w)[None, :])[..., None]
            acc[y0 * 4: y0 * 4 + bh, x0 * 4: x0 * 4 + bw] += part * wt
            wsum[y0 * 4: y0 * 4 + bh, x0 * 4: x0 * 4 + bw] += wt
        out = acc / np.maximum(wsum, 1e-6)
        return Image.fromarray(np.clip(out + 0.5, 0, 255).astype(np.uint8))
    finally:
        hook.remove()


# ----------------------------------------------------------------------------- framing

@dataclass
class Settings:
    mode: str = "print"            # "print" (inches at a DPI) or "scale" (factor)
    print_size: str = "auto"       # a PRESETS key, "auto" (largest preset matching the image ratio), or "WxH" like "16x20"
    unit: str = "in"               # "in" or "cm" for a custom WxH
    dpi: int = 300
    scale: float = 4.0             # used in "scale" mode
    framing: str = "fill"          # "fill" (crop to the print ratio), "fit" (add margin), "keep" (no crop, largest print that fits)
    crop_anchor: str = "center"    # "center", "top", "bottom", "left", "right"
    margin: str = "white"          # "fit" margin: "white", "black", "transparent", "edge"
    output_format: str = "png"     # "png" or "jpg"
    jpeg_quality: int = 95
    seed: int = 0
    block: int = 1024              # source pixels per AI block; lower it if the GPU runs out of memory
    batch: int = 0                 # 64 px tiles per GPU call; 0 = pick from GPU memory
    comparison: bool = True


@dataclass
class Plan:
    name: str
    src_w: int
    src_h: int
    crop_box: tuple
    canvas_w: int
    canvas_h: int
    target_w: int
    target_h: int
    print_w_in: float = None
    print_h_in: float = None
    notes: list = field(default_factory=list)

    @property
    def needed_scale(self):
        return max(self.target_w / self.canvas_w, self.target_h / self.canvas_h)

    @property
    def uses_ai(self):
        return self.target_w > self.canvas_w or self.target_h > self.canvas_h

    def ai_detail_dpi(self):
        if self.print_w_in is None:
            return None
        return min(self.canvas_w * 4 / self.print_w_in, self.canvas_h * 4 / self.print_h_in)


def load_image(path):
    """Opens with camera orientation applied; returns (image in RGB or RGBA, ICC profile bytes or None)."""
    img = Image.open(path)
    icc = img.info.get("icc_profile")
    img = ImageOps.exif_transpose(img)
    has_alpha = img.mode in ("RGBA", "LA", "PA") or (img.mode == "P" and "transparency" in img.info)
    if img.mode in ("I;16", "I;16B", "I", "F"):
        arr = np.asarray(img, dtype=np.float32)
        img = Image.fromarray((arr / max(arr.max(), 1) * 255).astype(np.uint8))
    img = img.convert("RGBA" if has_alpha else "RGB")
    if has_alpha and img.getextrema()[3][0] == 255:
        img = img.convert("RGB")  # alpha channel present but fully opaque
    return img, icc


def _parse_size(text, unit):
    w, h = (float(v) for v in text.lower().replace(" ", "").split("x"))
    if unit == "cm":
        w, h = w / 2.54, h / 2.54
    return w, h


def plan_for(name, img, s: Settings) -> Plan:
    w, h = img.size
    notes = []
    if s.mode == "scale":
        tw, th = round(w * s.scale), round(h * s.scale)
        return Plan(name, w, h, (0, 0, w, h), w, h, tw, th, notes=notes)

    ratio = min(w, h) / max(w, h)
    if s.print_size == "auto":
        matches = [k for k, (a, b) in PRESETS.items() if abs(a / b - ratio) < 0.02 and b <= 36]
        key = max(matches, key=lambda k: PRESETS[k][1]) if matches else "16x20"
        if not matches:
            notes.append(f"No preset matches the {w}x{h} ratio; using 16x20 with {s.framing}.")
        short, long_ = PRESETS[key]
    elif s.print_size in PRESETS:
        short, long_ = PRESETS[s.print_size]
    else:
        a, b = _parse_size(s.print_size, s.unit)
        short, long_ = min(a, b), max(a, b)
    landscape = w > h
    pw_in, ph_in = (long_, short) if landscape else (short, long_)

    if s.framing == "keep":
        # largest print of that size class that fits without cropping
        k = min(pw_in / w, ph_in / h)
        pw_in, ph_in = w * k, h * k
        plan = Plan(name, w, h, (0, 0, w, h), w, h, round(pw_in * s.dpi), round(ph_in * s.dpi), pw_in, ph_in, notes)
        _warn_soft(plan, s)
        return plan

    target_ratio = pw_in / ph_in
    if s.framing == "fill":
        if w / h > target_ratio:
            cw, ch = round(h * target_ratio), h
        else:
            cw, ch = w, round(w / target_ratio)
        ax = {"left": 0.0, "right": 1.0}.get(s.crop_anchor, 0.5)
        ay = {"top": 0.0, "bottom": 1.0}.get(s.crop_anchor, 0.5)
        x0 = round((w - cw) * ax)
        y0 = round((h - ch) * ay)
        box = (x0, y0, x0 + cw, y0 + ch)
        if (cw, ch) != (w, h):
            lost = 100 * (1 - cw * ch / (w * h))
            notes.append(f"Crops {lost:.0f}% of the image to fit {pw_in:g}x{ph_in:g} in.")
    else:  # fit
        if w / h > target_ratio:
            cw, ch = w, round(w / target_ratio)
        else:
            cw, ch = round(h * target_ratio), h
        box = (0, 0, w, h)
    tw, th = round(pw_in * s.dpi), round(ph_in * s.dpi)
    plan = Plan(name, w, h, box, cw, ch, tw, th, pw_in, ph_in, notes)
    _warn_soft(plan, s)
    return plan


def _warn_soft(plan, s):
    """Says plainly when most of the print's pixels will come from interpolation, not the AI."""
    dpi = plan.ai_detail_dpi()
    if plan.uses_ai and dpi is not None and dpi < s.dpi * 0.5:
        plan.notes.append(f"WARNING: AI detail is only about {round(dpi)} DPI at this size; "
                          f"expect a soft print up close. A smaller print or a larger source works better.")


def prepare_canvas(img, plan: Plan, s: Settings) -> Image.Image:
    if s.framing == "fit" and s.mode == "print" and (plan.canvas_w, plan.canvas_h) != (plan.src_w, plan.src_h):
        if s.margin == "transparent":
            canvas = Image.new("RGBA", (plan.canvas_w, plan.canvas_h), (0, 0, 0, 0))
        else:
            if s.margin == "edge":
                a = np.asarray(img.convert("RGB"))
                border = np.concatenate([a[0], a[-1], a[:, 0], a[:, -1]])
                color = tuple(int(v) for v in border.mean(axis=0))
            else:
                color = (255, 255, 255) if s.margin == "white" else (0, 0, 0)
            canvas = Image.new(img.mode, (plan.canvas_w, plan.canvas_h), color + ((255,) if img.mode == "RGBA" else ()))
        canvas.paste(img, ((plan.canvas_w - plan.src_w) // 2, (plan.canvas_h - plan.src_h) // 2))
        return canvas
    return img.crop(plan.crop_box)


def bleed_transparent(rgba: Image.Image) -> Image.Image:
    """Gives fully transparent pixels the colour of the nearest visible pixel (avoids dark halos)."""
    from scipy.ndimage import distance_transform_edt
    a = np.asarray(rgba)
    alpha = a[..., 3]
    if alpha.min() > 0 or alpha.max() == 0:
        return rgba.convert("RGB")
    _, (iy, ix) = distance_transform_edt(alpha == 0, return_indices=True)
    return Image.fromarray(a[..., :3][iy, ix])


# ----------------------------------------------------------------------------- run

def output_name(name, plan: Plan, s: Settings):
    base = "".join(c if c.isalnum() or c in "-_" else "_" for c in os.path.splitext(name)[0]).strip("_")[:48] or "Artwork"
    f = round(plan.needed_scale, 1)
    factor = f"{int(f)}x" if f == int(f) else f"{f}x"
    dpi = f"_{s.dpi}DPI" if s.mode == "print" else ""
    ext = "jpg" if s.output_format == "jpg" else "png"
    return f"{base}_{factor}_{plan.target_w}x{plan.target_h}{dpi}.{ext}"


def process(model, path, out_dir, s: Settings, log=print):
    """Upscales one image to its print target. Returns a result dict for the summary table."""
    t0 = time.time()
    name = os.path.basename(path)
    img, icc = load_image(path)
    plan = plan_for(name, img, s)
    canvas = prepare_canvas(img, plan, s)
    has_alpha = canvas.mode == "RGBA"
    rgb = bleed_transparent(canvas) if has_alpha else canvas

    if plan.uses_ai:
        last = [0.0]

        def prog(d, t):
            if time.time() - last[0] > 5 or d == t:
                last[0] = time.time()
                log(f"   AI {100 * d // t}%")
        ai = ai_upscale_4x(model, rgb, block=s.block, seed=s.seed, progress=prog, batch=s.batch or default_batch())
        method = "AuraSR 4x"
        if (ai.width, ai.height) != (plan.target_w, plan.target_h):
            ai = ai.resize((plan.target_w, plan.target_h), Image.LANCZOS)
            beyond = plan.needed_scale / 4
            method += f", then Lanczos {beyond:.2f}x" if beyond > 1.01 else ", then downsampled"
        out = ai
    else:
        out = rgb.resize((plan.target_w, plan.target_h), Image.LANCZOS)
        method = "Lanczos only (no enlargement needed)"

    if has_alpha:
        alpha = canvas.getchannel("A").resize((plan.target_w, plan.target_h), Image.LANCZOS)
        out = out.convert("RGBA")
        out.putalpha(alpha)

    fname = output_name(name, plan, s)
    dst = os.path.join(out_dir, fname)
    save_kw = {"dpi": (s.dpi, s.dpi)}
    if icc:
        save_kw["icc_profile"] = icc
    if s.output_format == "jpg":
        if out.mode == "RGBA":
            flat = Image.new("RGB", out.size, (255, 255, 255))
            flat.paste(out, mask=out.getchannel("A"))
            out = flat
        out.save(dst, "JPEG", quality=s.jpeg_quality, subsampling=0, optimize=True, **save_kw)
    else:
        out.save(dst, "PNG", compress_level=6, **save_kw)

    compare_path = None
    if s.comparison:
        compare_path = os.path.join(out_dir, "compare", os.path.splitext(fname)[0] + "_compare.jpg")
        os.makedirs(os.path.dirname(compare_path), exist_ok=True)
        comparison_sheet(rgb, out, plan).save(compare_path, quality=92)

    ai_dpi = plan.ai_detail_dpi() if plan.uses_ai else None
    res = {
        "file": name, "output": fname, "source": f"{plan.src_w}x{plan.src_h}",
        "output_px": f"{plan.target_w}x{plan.target_h}",
        "print": f"{plan.print_w_in:g}x{plan.print_h_in:g} in @ {s.dpi} DPI" if plan.print_w_in else "-",
        "ai_detail_dpi": round(ai_dpi) if ai_dpi else "-",
        "method": method, "size_mb": round(os.path.getsize(dst) / 2**20, 1),
        "seconds": round(time.time() - t0, 1), "notes": " ".join(plan.notes), "compare": compare_path,
    }
    return res


def comparison_sheet(src_rgb: Image.Image, out: Image.Image, plan: Plan, side=640) -> Image.Image:
    """100% crop from the centre of the output, next to the same area enlarged with Lanczos."""
    out_rgb = out.convert("RGB")
    W, H = out_rgb.size
    side = min(side, W, H)
    l, t = (W - side) // 2, (H - side) // 2
    after = out_rgb.crop((l, t, l + side, t + side))
    sx, sy = src_rgb.width / W, src_rgb.height / H
    box = (l * sx, t * sy, (l + side) * sx, (t + side) * sy)
    before = src_rgb.resize((side, side), Image.LANCZOS, box=box)
    sheet = Image.new("RGB", (side * 2 + 12, side + 36), (18, 17, 16))
    sheet.paste(before, (0, 36))
    sheet.paste(after, (side + 12, 36))
    d = ImageDraw.Draw(sheet)
    d.text((8, 8), "ORIGINAL, Lanczos enlargement (100% crop)", fill=(242, 238, 230), font=_font(18))
    d.text((side + 20, 8), "AURASR OUTPUT (100% crop)", fill=(242, 238, 230), font=_font(18))
    return sheet


def _font(size):
    from PIL import ImageFont
    try:
        return ImageFont.load_default(size=size)  # Pillow 10.1+
    except TypeError:
        return ImageFont.load_default()


def find_images(folder):
    found = []
    for root, _, files in os.walk(folder):
        for f in sorted(files):
            if os.path.splitext(f)[1].lower() in IMAGE_EXTS and not f.startswith("."):
                found.append(os.path.join(root, f))
    return found


def preview_plans(paths, s: Settings, max_side=360):
    """Contact sheet of every image with the crop/fit frame drawn on, before any processing."""
    tiles = []
    for p in paths:
        img, _ = load_image(p)
        plan = plan_for(os.path.basename(p), img, s)
        k = max_side / max(img.size)
        thumb = img.convert("RGB").resize((max(1, round(img.width * k)), max(1, round(img.height * k))))
        d = ImageDraw.Draw(thumb)
        if s.framing == "fill" and s.mode == "print":
            x0, y0, x1, y1 = (v * k for v in plan.crop_box)
            dim = Image.new("RGB", thumb.size, (0, 0, 0))
            mask = Image.new("L", thumb.size, 150)
            ImageDraw.Draw(mask).rectangle((x0, y0, x1 - 1, y1 - 1), fill=0)
            thumb = Image.composite(dim, thumb, mask)
            ImageDraw.Draw(thumb).rectangle((x0, y0, x1 - 1, y1 - 1), outline=(234, 100, 64), width=3)
        label = f"{plan.target_w}x{plan.target_h}px" + (f" | AI detail {round(plan.ai_detail_dpi())} DPI" if plan.uses_ai and plan.print_w_in else "")
        cell = Image.new("RGB", (max_side, max_side + 44), (18, 17, 16))
        cell.paste(thumb, ((max_side - thumb.width) // 2, (max_side - thumb.height) // 2))
        cd = ImageDraw.Draw(cell)
        cd.text((6, max_side + 4), os.path.basename(p)[:44], fill=(242, 238, 230), font=_font(15))
        cd.text((6, max_side + 23), label, fill=(184, 177, 166), font=_font(14))
        tiles.append((cell, plan))
    cols = min(3, max(1, len(tiles)))
    rows = math.ceil(len(tiles) / cols)
    sheet = Image.new("RGB", (cols * (max_side + 8), rows * (max_side + 52)), (10, 10, 10))
    for i, (cell, _) in enumerate(tiles):
        sheet.paste(cell, ((i % cols) * (max_side + 8), (i // cols) * (max_side + 52)))
    return sheet, [p for _, p in tiles]


def zip_folder(folder, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as z:  # PNG/JPEG are already compressed
        for root, _, files in os.walk(folder):
            for f in files:
                full = os.path.join(root, f)
                z.write(full, os.path.relpath(full, folder))
    return zip_path


CREDIT = ("Upscaled with AuraSR v2 by fal.ai (https://huggingface.co/fal/AuraSR-v2), "
          "licensed CC BY-SA 4.0 (https://creativecommons.org/licenses/by-sa/4.0/).")

In [ ]:
%%writefile printscale_server.py
# PrintScale AuraSR server: lets the PrintScale Android app use this notebook's GPU.
# The app uploads its prepared artwork, this server runs AuraSR 4x, the app downloads the result
# and does everything else (print sizing, DPI, colour profile, file export) on the phone.
#
# API (every request needs the header X-PrintScale-Key):
#   GET    /health            -> {"printscale": 1, "model", "device", "gpu", "max_pixels"}
#   POST   /jobs              body = PNG or JPEG  -> {"id"}
#   GET    /jobs/<id>         -> {"state": queued|running|done|failed|cancelled, "done", "total", "error"}
#   GET    /jobs/<id>/result  -> 4x PNG (RGBA if the upload had transparency)
#   DELETE /jobs/<id>         -> cancels and deletes the job
# Jobs are asynchronous because tunnels close requests that wait more than about 100 s.

import html
import io
import json
import os
import queue
import re
import secrets
import shutil
import subprocess
import tempfile
import threading
import time
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import quote

from PIL import Image

import printscale_aurasr as ps

MAX_UPLOAD_BYTES = 200 * 2**20
MAX_PIXELS = 4096 * 4096      # same limit as the app's planner
JOB_TTL_SECONDS = 3600


class Job:
    def __init__(self, path):
        self.id = uuid.uuid4().hex
        self.state = "queued"
        self.done = 0
        self.total = 0
        self.error = None
        self.input = path
        self.output = None
        self.created = time.time()
        self.cancel = threading.Event()

    def json(self):
        return {"id": self.id, "state": self.state, "done": self.done, "total": self.total, "error": self.error}


class PrintScaleServer:
    def __init__(self, model, device, port=8765, key=None, log=print):
        self.model, self.device, self.port, self.log = model, device, port, log
        self.key = key or secrets.token_urlsafe(18)
        self.workdir = tempfile.mkdtemp(prefix="printscale_")
        self.jobs = {}
        self.queue = queue.Queue()
        self.lock = threading.Lock()
        self.httpd = None
        self.tunnel = None
        self.public_url = None
        import torch
        self.gpu = torch.cuda.get_device_name(0) if device == "cuda" else ""

    # ------------------------------------------------------------------ GPU worker (one job at a time)
    def _worker(self):
        import torch
        while True:
            job = self.queue.get()
            if job.cancel.is_set():
                continue
            job.state = "running"
            t0 = time.time()
            try:
                img = Image.open(job.input)
                img.load()
                alpha = img.getchannel("A") if img.mode in ("RGBA", "LA") else None
                rgb = img.convert("RGB")

                def progress(d, t):
                    job.done, job.total = d, t

                batch, block = ps.default_batch(), 512  # 512 px blocks keep memory low and progress smooth
                for attempt in range(4):
                    try:
                        out = ps.ai_upscale_4x(self.model, rgb, block=block, batch=batch, progress=progress,
                                               cancelled=job.cancel.is_set)
                        break
                    except torch.cuda.OutOfMemoryError:
                        torch.cuda.empty_cache()
                        batch = max(1, batch // 2)
                        self.log(f"GPU memory full, retrying with batch {batch}")
                else:
                    raise RuntimeError("Not enough GPU memory for this image.")
                if alpha is not None:
                    out.putalpha(alpha.resize(out.size, Image.BICUBIC))
                job.output = os.path.join(self.workdir, job.id + "_out.png")
                out.save(job.output, "PNG", compress_level=4)
                job.state = "done"
                self.log(f"job {job.id[:8]}: {rgb.width}x{rgb.height} -> {out.width}x{out.height} in {time.time() - t0:.0f}s")
            except InterruptedError:
                job.state = "cancelled"
            except Exception as e:
                job.state, job.error = "failed", f"{type(e).__name__}: {e}"
                self.log(f"job {job.id[:8]} failed: {job.error}")
            finally:
                if self.device == "cuda":
                    torch.cuda.empty_cache()
                if os.path.exists(job.input):
                    os.remove(job.input)

    def _janitor(self):
        while True:
            time.sleep(300)
            now = time.time()
            with self.lock:
                old = [j for j in self.jobs.values() if now - j.created > JOB_TTL_SECONDS]
                for j in old:
                    self._delete(j)

    def _delete(self, job):
        job.cancel.set()
        for p in (job.input, job.output):
            if p and os.path.exists(p):
                os.remove(p)
        self.jobs.pop(job.id, None)

    # ------------------------------------------------------------------ HTTP
    def _handler(self):
        server = self

        class Handler(BaseHTTPRequestHandler):
            protocol_version = "HTTP/1.1"

            def log_message(self, *args):
                pass

            def _send_json(self, code, obj):
                body = json.dumps(obj).encode()
                self.send_response(code)
                self.send_header("Content-Type", "application/json")
                self.send_header("Content-Length", str(len(body)))
                self.end_headers()
                self.wfile.write(body)

            def _authorized(self):
                got = self.headers.get("X-PrintScale-Key") or ""
                if not got:
                    m = re.search(r"[?&]key=([^&]+)", self.path)
                    got = m.group(1) if m else ""
                return secrets.compare_digest(got, server.key)

            def _job(self, jid):
                job = server.jobs.get(jid)
                if job is None:
                    self._send_json(404, {"error": "No such job. The notebook may have restarted."})
                return job

            def do_GET(self):
                path = self.path.split("?")[0]
                if path in ("/", "/connect"):
                    return self._connect_page()
                if not self._authorized():
                    return self._send_json(401, {"error": "Wrong key."})
                if path == "/health":
                    return self._send_json(200, {"printscale": 1, "model": "AuraSR v2", "device": server.device,
                                                 "gpu": server.gpu, "max_pixels": MAX_PIXELS})
                m = re.fullmatch(r"/jobs/([0-9a-f]{32})(/result)?", path)
                if not m:
                    return self._send_json(404, {"error": "Unknown path."})
                job = self._job(m.group(1))
                if job is None:
                    return
                if not m.group(2):
                    return self._send_json(200, job.json())
                if job.state != "done":
                    return self._send_json(409, {"error": f"Job is {job.state}."})
                size = os.path.getsize(job.output)
                self.send_response(200)
                self.send_header("Content-Type", "image/png")
                self.send_header("Content-Length", str(size))
                self.end_headers()
                with open(job.output, "rb") as f:
                    shutil.copyfileobj(f, self.wfile, 1 << 20)

            def do_POST(self):
                if not self._authorized():
                    return self._send_json(401, {"error": "Wrong key."})
                if self.path.split("?")[0] != "/jobs":
                    return self._send_json(404, {"error": "Unknown path."})
                length = int(self.headers.get("Content-Length") or 0)
                if length <= 0:
                    return self._send_json(411, {"error": "Empty upload."})
                if length > MAX_UPLOAD_BYTES:
                    return self._send_json(413, {"error": "Upload is larger than 200 MB."})
                path = os.path.join(server.workdir, uuid.uuid4().hex + "_in")
                left = length
                with open(path, "wb") as f:
                    while left > 0:
                        chunk = self.rfile.read(min(1 << 20, left))
                        if not chunk:
                            break
                        f.write(chunk)
                        left -= len(chunk)
                if left:
                    os.remove(path)
                    return self._send_json(400, {"error": "Upload was cut off."})
                try:
                    with Image.open(path) as im:
                        w, h = im.size
                        im.verify()
                except Exception:
                    os.remove(path)
                    return self._send_json(400, {"error": "The upload is not an image."})
                if w * h > MAX_PIXELS:
                    os.remove(path)
                    return self._send_json(413, {"error": f"{w}x{h} is over the server's {MAX_PIXELS // 10**6} MP limit."})
                job = Job(path)
                with server.lock:
                    server.jobs[job.id] = job
                server.queue.put(job)
                server.log(f"job {job.id[:8]}: received {w}x{h}")
                self._send_json(200, {"id": job.id})

            def do_DELETE(self):
                if not self._authorized():
                    return self._send_json(401, {"error": "Wrong key."})
                m = re.fullmatch(r"/jobs/([0-9a-f]{32})", self.path.split("?")[0])
                job = server.jobs.get(m.group(1)) if m else None
                if job:
                    with server.lock:
                        server._delete(job)
                self._send_json(200, {"ok": True})

            def _connect_page(self):
                # Opened by scanning the QR code on a phone: one tap hands the link to the app.
                if not self._authorized():
                    body = b"<h1>PrintScale server</h1><p>Open the full link from the notebook.</p>"
                    self.send_response(401)
                else:
                    base = server.public_url or f"http://localhost:{server.port}"
                    deep = f"printscale://connect?server={quote(base, safe='')}&key={quote(server.key, safe='')}"
                    full = f"{base}/?key={server.key}"
                    body = f"""<!doctype html><html><head><meta name="viewport" content="width=device-width,initial-scale=1">
<title>PrintScale server</title></head><body style="font-family:sans-serif;background:#121110;color:#f2eee6;padding:24px">
<h1 style="font-weight:500">PrintScale cloud GPU</h1>
<p>AuraSR v2 is running on {html.escape(server.gpu or server.device)}.</p>
<p><a href="{html.escape(deep)}" style="display:inline-block;background:#ea6440;color:#1a0f0b;padding:14px 20px;border-radius:10px;text-decoration:none;font-weight:600">Open in the PrintScale app</a></p>
<p>Or copy this link and paste it in the app:</p><p style="word-break:break-all;color:#b8b1a6">{html.escape(full)}</p>
</body></html>""".encode()
                    self.send_response(200)
                self.send_header("Content-Type", "text/html; charset=utf-8")
                self.send_header("Content-Length", str(len(body)))
                self.end_headers()
                self.wfile.write(body)

        return Handler

    def start(self):
        threading.Thread(target=self._worker, daemon=True).start()
        threading.Thread(target=self._janitor, daemon=True).start()
        self.httpd = ThreadingHTTPServer(("127.0.0.1", self.port), self._handler())
        threading.Thread(target=self.httpd.serve_forever, daemon=True).start()
        return self

    def start_tunnel(self, binary="./cloudflared", timeout=60):
        """Opens a free Cloudflare quick tunnel (no account) and returns the public https URL."""
        if not os.path.exists(binary):
            url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
            subprocess.run(["wget", "-q", "-O", binary, url], check=True)
            os.chmod(binary, 0o755)
        self.tunnel = subprocess.Popen([binary, "tunnel", "--url", f"http://127.0.0.1:{self.port}", "--no-autoupdate"],
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        deadline = time.time() + timeout
        for line in self.tunnel.stdout:
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if m:
                self.public_url = m.group(0)
                threading.Thread(target=lambda: [None for _ in self.tunnel.stdout], daemon=True).start()  # keep draining
                return self.public_url
            if time.time() > deadline:
                break
        raise RuntimeError("The Cloudflare tunnel did not start. Run this cell again.")

    @property
    def link(self):
        return f"{self.public_url or f'http://localhost:{self.port}'}/?key={self.key}"

    def status_line(self):
        states = [j.state for j in list(self.jobs.values())]
        running = [j for j in self.jobs.values() if j.state == "running"]
        now = f" | running: {100 * running[0].done // max(1, running[0].total)}%" if running else ""
        return f"{time.strftime('%H:%M:%S')} jobs: {len(states)} ({states.count('done')} done){now}"

In [ ]:
import importlib, sys
sys.path.insert(0, os.getcwd())
import printscale_aurasr as ps
importlib.reload(ps)
print("ready")

In [ ]:
# 5. Start the server and show the link (keep this running)
import time
import qrcode
from IPython.display import display, clear_output
import printscale_server as srv
importlib.reload(srv)

model, device = ps.load_aurasr()
server = srv.PrintScaleServer(model, device, port=int(os.environ.get("PRINTSCALE_PORT", 8765))).start()
if os.environ.get("PRINTSCALE_NO_TUNNEL") == "1":
    print("Test mode: no tunnel, local link only.")
else:
    print("Opening a Cloudflare tunnel...")
    server.start_tunnel()
connect_page = server.link.replace("/?key=", "/connect?key=")

def show():
    clear_output(wait=True)
    print("PrintScale cloud GPU is running on", (server.gpu or device).upper())
    print()
    print("Paste this link in the app (Home > Cloud GPU):")
    print(server.link)
    print()
    print("Or scan with your phone's camera, then tap 'Open in the PrintScale app':")
    display(qrcode.make(connect_page).resize((280, 280)))
    print(server.status_line())

show()
if os.environ.get("PRINTSCALE_NO_TUNNEL") != "1":
    try:
        while True:          # keeps the cell busy so the notebook stays alive; stop it to shut the server down
            time.sleep(30)
            show()
    except KeyboardInterrupt:
        print("Server stopped.")